# 04 — Fairness Mitigation

This notebook applies fairness-aware retraining to the baseline model  
and compares the accuracy–fairness trade-off.

**Approach:** Fairlearn's `ExponentiatedGradient` with `BoundedGroupLoss`  
wraps around the same XGBoost estimator and enforces a constraint that  
prediction loss should be roughly equal across demographic groups.

**Fallback:** If `BoundedGroupLoss` does not converge well, we also implement  
a simpler sample-reweighting approach as an alternative.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from fairlearn.reductions import ExponentiatedGradient, BoundedGroupLoss, ZeroOneLoss
from fairlearn.metrics import MetricFrame
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
FIG_DIR = '../figures'

FEATURE_COLS = ['trip_miles', 'trip_seconds', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour']
TARGET_COL = 'fare'
SENSITIVE_COL = 'majority_race'

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

In [ ]:
# ── Load train and test sets ─────────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_set.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_with_predictions.csv'))

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
sens_train = train_df[SENSITIVE_COL]

X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]
sens_test = test_df[SENSITIVE_COL]

# Baseline predictions (from notebook 02) for comparison
y_pred_baseline = test_df['predicted_fare'].values

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 4.1 — Fairlearn Mitigation: Exponentiated Gradient

We use `BoundedGroupLoss` which constrains the maximum loss in any single group  
to be close to the average loss. This prevents the model from being significantly  
worse for one group at the expense of another.

> **Note on compute time:** ExponentiatedGradient trains multiple rounds of the  
> base estimator. With large datasets, this can take several minutes.  
> Consider subsampling training data if needed (e.g., 100K–200K rows).

In [ ]:
# ── Subsample training data if needed for faster iteration ────────────────
MAX_TRAIN_ROWS = 200_000

if len(X_train) > MAX_TRAIN_ROWS:
    sample_idx = np.random.RandomState(42).choice(len(X_train), MAX_TRAIN_ROWS, replace=False)
    X_train_sub = X_train.iloc[sample_idx]
    y_train_sub = y_train.iloc[sample_idx]
    sens_train_sub = sens_train.iloc[sample_idx]
    print(f'Subsampled training data to {MAX_TRAIN_ROWS:,} rows for mitigation')
else:
    X_train_sub = X_train
    y_train_sub = y_train
    sens_train_sub = sens_train
    print(f'Using full training data: {len(X_train):,} rows')

In [ ]:
# ── Train mitigated model ────────────────────────────────────────────────────
base_estimator = XGBRegressor(
    n_estimators=100,     # fewer trees since EG trains multiple rounds
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)

constraint = BoundedGroupLoss(ZeroOneLoss(), upper_bound=0.1)

mitigator = ExponentiatedGradient(
    estimator=base_estimator,
    constraints=constraint,
    max_iter=50,
    nu=1e-4,
)

print('Training mitigated model (this may take a few minutes)...')
mitigator.fit(X_train_sub, y_train_sub, sensitive_features=sens_train_sub)
print('Done.')

In [ ]:
y_pred_mitigated = mitigator.predict(X_test)
print(f'Generated {len(y_pred_mitigated):,} mitigated predictions')

## 4.2 — Alternative: Sample Reweighting

If ExponentiatedGradient didn't converge well, this simpler approach  
up-weights underrepresented or worse-served groups during training.  
Run this section as a comparison or fallback.

In [ ]:
# ── Compute sample weights inversely proportional to group size ────────────
group_counts = sens_train.value_counts()
max_count = group_counts.max()
weight_map = {g: max_count / count for g, count in group_counts.items()}
sample_weights = sens_train.map(weight_map).values

print('Sample weight map:', {k: round(v, 3) for k, v in weight_map.items()})

model_reweighted = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model_reweighted.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_reweighted = model_reweighted.predict(X_test)
print('Reweighted model trained.')

## 4.3 — Comparative Evaluation

We now compare three models:
1. **Baseline** — no fairness constraints
2. **Fairlearn (EG)** — BoundedGroupLoss constraint
3. **Reweighted** — sample weight balancing

In [ ]:
def compute_group_metrics(y_true, y_pred, groups):
    """Compute per-group and overall regression metrics."""
    results = {}
    for group in sorted(groups.unique()):
        mask = groups == group
        yt, yp = y_true[mask], y_pred[mask]
        results[group] = {
            'N': int(mask.sum()),
            'MAE': mean_absolute_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'Mean Error': (yp - yt).mean(),
        }
    # Overall
    results['OVERALL'] = {
        'N': len(y_true),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'Mean Error': (y_pred - y_true).mean(),
    }
    return pd.DataFrame(results).T


y_true_arr = y_test.values
sens_arr = sens_test.values

metrics_baseline = compute_group_metrics(y_true_arr, y_pred_baseline, sens_test)
metrics_mitigated = compute_group_metrics(y_true_arr, y_pred_mitigated, sens_test)
metrics_reweighted = compute_group_metrics(y_true_arr, y_pred_reweighted, sens_test)

print('=== BASELINE ===')
print(metrics_baseline.round(3).to_string())
print('\n=== FAIRLEARN (Exponentiated Gradient) ===')
print(metrics_mitigated.round(3).to_string())
print('\n=== REWEIGHTED ===')
print(metrics_reweighted.round(3).to_string())

In [ ]:
# ── Fairness gap: max RMSE difference between groups ─────────────────────────
def fairness_gap(metrics_df):
    groups_only = metrics_df.drop('OVERALL', errors='ignore')
    return groups_only['RMSE'].max() - groups_only['RMSE'].min()

gap_baseline = fairness_gap(metrics_baseline)
gap_mitigated = fairness_gap(metrics_mitigated)
gap_reweighted = fairness_gap(metrics_reweighted)

print('=== Fairness Gap (max RMSE - min RMSE across groups) ===')
print(f'  Baseline:   {gap_baseline:.4f}')
print(f'  Fairlearn:  {gap_mitigated:.4f}  ({(1 - gap_mitigated/gap_baseline)*100:+.1f}% reduction)')
print(f'  Reweighted: {gap_reweighted:.4f}  ({(1 - gap_reweighted/gap_baseline)*100:+.1f}% reduction)')

## 4.4 — Visualization: Baseline vs Mitigated

In [ ]:
# ── Grouped bar chart comparing RMSE across models ────────────────────────────
groups = [g for g in metrics_baseline.index if g != 'OVERALL']

x = np.arange(len(groups))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, metrics_baseline.loc[groups, 'RMSE'], width, label='Baseline', color='#c44e52')
ax.bar(x,         metrics_mitigated.loc[groups, 'RMSE'], width, label='Fairlearn (EG)', color='#4c72b0')
ax.bar(x + width, metrics_reweighted.loc[groups, 'RMSE'], width, label='Reweighted', color='#55a868')

ax.set_ylabel('RMSE ($)')
ax.set_title('RMSE by Demographic Group: Baseline vs Mitigated Models')
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'mitigation_rmse_comparison.png'), dpi=150)
plt.show()

In [ ]:
# ── Mean error comparison ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, metrics_baseline.loc[groups, 'Mean Error'], width, label='Baseline', color='#c44e52')
ax.bar(x,         metrics_mitigated.loc[groups, 'Mean Error'], width, label='Fairlearn (EG)', color='#4c72b0')
ax.bar(x + width, metrics_reweighted.loc[groups, 'Mean Error'], width, label='Reweighted', color='#55a868')

ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_ylabel('Mean Error ($)')
ax.set_title('Mean Prediction Error by Group: Baseline vs Mitigated')
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'mitigation_error_comparison.png'), dpi=150)
plt.show()

## 4.5 — Accuracy–Fairness Trade-off

In [ ]:
# ── Trade-off table ──────────────────────────────────────────────────────────
tradeoff = pd.DataFrame({
    'Model': ['Baseline', 'Fairlearn (EG)', 'Reweighted'],
    'Overall RMSE': [
        metrics_baseline.loc['OVERALL', 'RMSE'],
        metrics_mitigated.loc['OVERALL', 'RMSE'],
        metrics_reweighted.loc['OVERALL', 'RMSE'],
    ],
    'Overall MAE': [
        metrics_baseline.loc['OVERALL', 'MAE'],
        metrics_mitigated.loc['OVERALL', 'MAE'],
        metrics_reweighted.loc['OVERALL', 'MAE'],
    ],
    'Fairness Gap (RMSE)': [gap_baseline, gap_mitigated, gap_reweighted],
})

tradeoff['Accuracy Cost (RMSE %)'] = (
    (tradeoff['Overall RMSE'] - tradeoff['Overall RMSE'].iloc[0])
    / tradeoff['Overall RMSE'].iloc[0] * 100
).round(2)

tradeoff['Fairness Gain (%)'] = (
    (1 - tradeoff['Fairness Gap (RMSE)'] / tradeoff['Fairness Gap (RMSE)'].iloc[0]) * 100
).round(2)

print('=== Accuracy–Fairness Trade-off ===')
print(tradeoff.round(4).to_string(index=False))

tradeoff.to_csv(os.path.join(DATA_DIR, 'tradeoff_summary.csv'), index=False)
print(f'\nSaved to {DATA_DIR}/tradeoff_summary.csv')

In [ ]:
# ── Scatter: overall RMSE vs fairness gap ────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

for _, row in tradeoff.iterrows():
    ax.scatter(row['Fairness Gap (RMSE)'], row['Overall RMSE'], s=120, zorder=3)
    ax.annotate(row['Model'], (row['Fairness Gap (RMSE)'], row['Overall RMSE']),
                textcoords='offset points', xytext=(8, 6), fontsize=10)

ax.set_xlabel('Fairness Gap (max − min group RMSE)')
ax.set_ylabel('Overall RMSE ($)')
ax.set_title('Accuracy–Fairness Trade-off')
ax.annotate('← Fairer', xy=(0.05, 0.02), xycoords='axes fraction', fontsize=9, color='green')
ax.annotate('More accurate ↓', xy=(0.85, 0.02), xycoords='axes fraction', fontsize=9, color='blue')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'tradeoff_scatter.png'), dpi=150)
plt.show()

## 4.6 — Fairlearn MetricFrame Summary

Using Fairlearn's built-in `MetricFrame` for a clean grouped summary.

In [ ]:
# Fairlearn MetricFrame for the mitigated model
mf = MetricFrame(
    metrics={
        'MAE': mean_absolute_error,
        'RMSE': lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
    },
    y_true=y_true_arr,
    y_pred=y_pred_mitigated,
    sensitive_features=sens_arr,
)

print('--- Fairlearn MetricFrame (Mitigated Model) ---')
print('Overall:')
print(mf.overall.round(3))
print('\nBy group:')
print(mf.by_group.round(3))
print('\nDifference (max − min):')
print(mf.difference().round(4))
print('\nRatio (min / max):')
print(mf.ratio().round(4))